# Notebook 3: SEC EDGAR Company Fundamentals

**Alpha Search — Real Data Only**

This notebook fetches real company financial data from the **SEC EDGAR** (Electronic Data Gathering, Analysis, and Retrieval) system using their free public API.

**Data Sources:**
- **SEC EDGAR API** (data.sec.gov) — Free, no API key needed
  - Company facts (XBRL data): revenue, net income, EPS, assets, liabilities
  - Proper User-Agent header required

**Companies Analyzed:**
| Ticker | CIK | Company |
|--------|-----|---------|
| AAPL | 0000320193 | Apple Inc. |
| MSFT | 0000789019 | Microsoft Corp. |
| TSLA | 0001318605 | Tesla Inc. |

**Metrics Extracted:**
- Annual Revenue
- Annual Net Income
- Earnings Per Share (EPS)
- Profit Margin (Net Income / Revenue)

**Important:** SEC EDGAR requires a proper `User-Agent` header with contact info.

In [ ]:
# Install Alpha Search
!pip install git+https://github.com/alpha-search/alpha-search.git -q

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

In [ ]:
# SEC EDGAR Fetcher Functions
# SEC requires a proper User-Agent header with contact information

HEADERS = {'User-Agent': 'AlphaSearch Research (research@alphasearch.io)'}

def get_cik(ticker):
    """Look up CIK (Central Index Key) for a given stock ticker."""
    url = 'https://www.sec.gov/files/company_tickers.json'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        for entry in data.values():
            if entry['ticker'].upper() == ticker.upper():
                return str(entry['cik_str']).zfill(10)
        raise ValueError(f'Ticker {ticker} not found in SEC database')
    except Exception as e:
        print(f"  ERROR looking up CIK for {ticker}: {e}")
        return None

def fetch_sec_facts(cik):
    """Fetch company facts (XBRL data) from SEC EDGAR. CIK must be 10-digit zero-padded."""
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    resp = None
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError as e:
        if resp is not None and resp.status_code == 403:
            print(f"  ACCESS DENIED (403): SEC may be blocking this IP. Try again later.")
        elif resp is not None and resp.status_code == 404:
            print(f"  NOT FOUND (404): No data for CIK {cik}")
        else:
            print(f"  HTTP ERROR: {e}")
        return None
    except Exception as e:
        print(f"  ERROR fetching SEC data for CIK {cik}: {e}")
        return None

def extract_fact(facts_data, taxonomy, tag_name):
    """Extract a specific XBRL fact (e.g., Revenue, NetIncome) from company facts."""
    try:
        units = facts_data['facts'][taxonomy][tag_name]['units']
        unit_key = 'USD' if 'USD' in units else list(units.keys())[0]
        records = units[unit_key]
        df = pd.DataFrame(records)
        df = df[df['form'].isin(['10-K', '10-Q'])].copy()
        df['end'] = pd.to_datetime(df['end'])
        df['val'] = pd.to_numeric(df['val'], errors='coerce')
        df = df.dropna(subset=['val'])
        df = df[df['form'] == '10-K'].copy()
        df = df.sort_values('end')
        return df.set_index('end')['val']
    except (KeyError, TypeError):
        return pd.Series()

print('SEC EDGAR fetcher functions defined.')

In [ ]:
# Fetch company facts for AAPL
print('Fetching SEC EDGAR data for Apple Inc. (AAPL)...')
print('=' * 60)

cik_aapl = get_cik('AAPL')
print(f'AAPL CIK: {cik_aapl}')

if cik_aapl:
    time.sleep(0.5)
    aapl_facts = fetch_sec_facts(cik_aapl)
    if aapl_facts:
        print(f'Successfully fetched AAPL facts. Keys: {list(aapl_facts.keys())}')
    else:
        print('ERROR: Could not fetch AAPL facts. SEC may be rate-limiting.')
        aapl_facts = None
else:
    aapl_facts = None
    print('ERROR: Could not look up AAPL CIK.')

In [ ]:
# Display Company Overview
print('APPLE INC. (AAPL) — COMPANY OVERVIEW')
print('=' * 60)

if aapl_facts and 'entityName' in aapl_facts:
    print(f"Company: {aapl_facts['entityName']}")
    print(f"CIK:     {aapl_facts.get('cik', 'N/A')}")
    
    # Show available taxonomies and facts
    if 'facts' in aapl_facts:
        taxonomies = list(aapl_facts['facts'].keys())
        print(f"\nAvailable Taxonomies: {taxonomies}")
        for tax in taxonomies[:2]:
            tags = list(aapl_facts['facts'][tax].keys())
            print(f"  [{tax}]: {len(tags)} facts (e.g., {', '.join(tags[:5])}...)")
else:
    print('No company data available. SEC API may be temporarily unavailable.')
    print('This is normal — the SEC API can be slow or rate-limit shared IPs.')

In [ ]:
# Extract and Plot Annual Revenue for AAPL
if aapl_facts and 'facts' in aapl_facts:
    print('Extracting AAPL Revenue...')
    
    # Try different revenue tags used by different filers
    revenue_tags = ['RevenueFromContractWithCustomerExcludingAssessedTax', 
                   'Revenues', 'SalesRevenueNet', 'RevenueNet']
    
    aapl_revenue = pd.Series()
    for tag in revenue_tags:
        aapl_revenue = extract_fact(aapl_facts, 'us-gaap', tag)
        if len(aapl_revenue) > 0:
            print(f'  Found revenue data via tag: {tag} ({len(aapl_revenue)} records)')
            break
    
    if len(aapl_revenue) == 0:
        # Try alternative taxonomies
        for tax in aapl_facts['facts'].keys():
            for tag in ['Revenues', 'RevenueFromContractWithCustomerExcludingAssessedTax']:
                aapl_revenue = extract_fact(aapl_facts, tax, tag)
                if len(aapl_revenue) > 0:
                    print(f'  Found revenue via {tax}/{tag} ({len(aapl_revenue)} records)')
                    break
            if len(aapl_revenue) > 0:
                break
    
    if len(aapl_revenue) > 0:
        # Convert to billions
        rev_b = aapl_revenue / 1e9
        
        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.bar(rev_b.index.year.astype(str), rev_b.values, 
                      color='#1f77b4', alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.set_title('Apple Inc. Annual Revenue', fontsize=16, fontweight='bold')
        ax.set_xlabel('Fiscal Year', fontsize=12)
        ax.set_ylabel('Revenue (Billions USD)', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for bar, val in zip(bars, rev_b.values):
            ax.text(bar.get_x() + bar.get_width()/2., val, 
                    f'${val:.0f}B', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        print(f'Real SEC data: AAPL revenue from {rev_b.index.min().year} to {rev_b.index.max().year}')
    else:
        print('ERROR: Could not extract revenue data. Tag names may differ in this filing.')
else:
    print('ERROR: No AAPL facts data available.')

In [ ]:
# Extract and Plot Annual Net Income for AAPL
if aapl_facts and 'facts' in aapl_facts:
    print('Extracting AAPL Net Income...')
    
    ni_tags = ['NetIncomeLoss', 'ProfitLoss', 'NetIncome']
    aapl_ni = pd.Series()
    for tax in aapl_facts['facts'].keys():
        for tag in ni_tags:
            aapl_ni = extract_fact(aapl_facts, tax, tag)
            if len(aapl_ni) > 0:
                print(f'  Found net income via {tax}/{tag} ({len(aapl_ni)} records)')
                break
        if len(aapl_ni) > 0:
            break
    
    if len(aapl_ni) > 0:
        ni_b = aapl_ni / 1e9
        
        fig, ax = plt.subplots(figsize=(12, 6))
        colors = ['green' if v >= 0 else 'red' for v in ni_b.values]
        bars = ax.bar(ni_b.index.year.astype(str), ni_b.values, 
                      color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.set_title('Apple Inc. Annual Net Income', fontsize=16, fontweight='bold')
        ax.set_xlabel('Fiscal Year', fontsize=12)
        ax.set_ylabel('Net Income (Billions USD)', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')
        ax.axhline(y=0, color='black', linewidth=0.5)
        
        for bar, val in zip(bars, ni_b.values):
            ax.text(bar.get_x() + bar.get_width()/2., val, 
                    f'${val:.0f}B', ha='center', 
                    va='bottom' if val >= 0 else 'top', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        print(f'Real SEC data: AAPL net income from {ni_b.index.min().year} to {ni_b.index.max().year}')
    else:
        print('ERROR: Could not extract net income data.')
else:
    print('ERROR: No AAPL facts data available.')

In [ ]:
# Calculate and Plot Profit Margin
if aapl_facts and 'facts' in aapl_facts and len(aapl_revenue) > 0 and len(aapl_ni) > 0:
    print('Calculating Profit Margin (Net Income / Revenue)...')
    
    # Align by index (year end dates)
    margin = (aapl_ni / aapl_revenue * 100).dropna()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(margin.index.year.astype(str), margin.values, 
            marker='o', linewidth=2.5, markersize=8, color='#2ca02c')
    ax.fill_between(range(len(margin)), margin.values, 0, 
                    alpha=0.2, color='#2ca02c')
    ax.set_title('Apple Inc. Net Profit Margin', fontsize=16, fontweight='bold')
    ax.set_xlabel('Fiscal Year', fontsize=12)
    ax.set_ylabel('Profit Margin (%)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=20, color='orange', linestyle='--', alpha=0.5, label='20% Benchmark')
    ax.legend()
    
    for i, (year, val) in enumerate(zip(margin.index.year, margin.values)):
        ax.text(i, val, f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    latest_margin = margin.iloc[-1]
    print(f'Real SEC data: Latest profit margin = {latest_margin:.1f}%')
    print(f'  Interpretation: Apple keeps {latest_margin:.1f} cents of every dollar as profit.')
else:
    print('ERROR: Cannot calculate margin — missing revenue or net income data.')

In [ ]:
# Extract and Plot EPS (Earnings Per Share)
if aapl_facts and 'facts' in aapl_facts:
    print('Extracting AAPL EPS...')
    
    eps_tags = ['EarningsPerShareDiluted', 'EarningsPerShareBasic', 
                'EarningsPerShareBasicAndDiluted']
    aapl_eps = pd.Series()
    for tax in aapl_facts['facts'].keys():
        for tag in eps_tags:
            aapl_eps = extract_fact(aapl_facts, tax, tag)
            if len(aapl_eps) > 0:
                print(f'  Found EPS via {tax}/{tag} ({len(aapl_eps)} records)')
                break
        if len(aapl_eps) > 0:
            break
    
    if len(aapl_eps) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(aapl_eps.index.year.astype(str), aapl_eps.values, 
                marker='s', linewidth=2.5, markersize=8, color='#9467bd')
        ax.fill_between(range(len(aapl_eps)), aapl_eps.values, 0, 
                        alpha=0.2, color='#9467bd')
        ax.set_title('Apple Inc. Earnings Per Share (EPS)', fontsize=16, fontweight='bold')
        ax.set_xlabel('Fiscal Year', fontsize=12)
        ax.set_ylabel('EPS (USD)', fontsize=12)
        ax.grid(True, alpha=0.3)
        
        for i, (year, val) in enumerate(zip(aapl_eps.index.year, aapl_eps.values)):
            ax.text(i, val, f'${val:.2f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        print(f'Real SEC data: Latest EPS = ${aapl_eps.iloc[-1]:.2f}')
    else:
        print('ERROR: Could not extract EPS data from filing.')
else:
    print('ERROR: No AAPL facts data available.')

In [ ]:
# Compare 3 Companies Side by Side
print('Fetching data for comparison: AAPL, MSFT, TSLA')
print('=' * 60)

tickers = {'AAPL': 'Apple', 'MSFT': 'Microsoft', 'TSLA': 'Tesla'}
company_data = {}

for ticker in tickers.keys():
    print(f'\n--- {ticker} ---')
    cik = get_cik(ticker)
    if cik:
        print(f'  CIK: {cik}')
        time.sleep(0.5)
        facts = fetch_sec_facts(cik)
        if facts and 'facts' in facts:
            # Extract revenue
            rev = pd.Series()
            for tax in facts['facts'].keys():
                for tag in ['RevenueFromContractWithCustomerExcludingAssessedTax', 
                           'Revenues', 'SalesRevenueNet']:
                    rev = extract_fact(facts, tax, tag)
                    if len(rev) > 0:
                        break
                if len(rev) > 0:
                    break
            
            # Extract net income
            ni = pd.Series()
            for tax in facts['facts'].keys():
                for tag in ['NetIncomeLoss', 'ProfitLoss']:
                    ni = extract_fact(facts, tax, tag)
                    if len(ni) > 0:
                        break
                if len(ni) > 0:
                    break
            
            company_data[ticker] = {'revenue': rev, 'net_income': ni}
            print(f'  Revenue: {len(rev)} years, Net Income: {len(ni)} years')
        else:
            print(f'  ERROR: Could not fetch facts for {ticker}')
    time.sleep(0.5)  # Be respectful to SEC servers

print('=' * 60)
fetched = [k for k, v in company_data.items() if len(v.get('revenue', [])) > 0]
print(f'Successfully fetched: {fetched}')

In [ ]:
# Side-by-Side Revenue Comparison
if len(company_data) > 0:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    bar_colors = {'AAPL': '#555555', 'MSFT': '#00A4EF', 'TSLA': '#CC0000'}
    
    for ticker, data in company_data.items():
        rev = data['revenue']
        if len(rev) > 0:
            rev_b = rev / 1e9
            years = rev_b.index.year
            ax.plot(years, rev_b.values, marker='o', linewidth=2.5, 
                    markersize=8, label=ticker, color=bar_colors.get(ticker, 'gray'))
    
    ax.set_title('Revenue Comparison: AAPL vs MSFT vs TSLA\n(Real SEC EDGAR Data)', 
                 fontsize=16, fontweight='bold')
    ax.set_xlabel('Fiscal Year', fontsize=12)
    ax.set_ylabel('Revenue (Billions USD)', fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print('\nLATEST ANNUAL REVENUE COMPARISON:')
    print('-' * 50)
    for ticker, data in company_data.items():
        rev = data['revenue']
        if len(rev) > 0:
            latest_rev = rev.iloc[-1] / 1e9
            latest_year = rev.index[-1].year
            print(f'  {ticker}: ${latest_rev:,.0f}B ({latest_year})')
    print('-' * 50)
else:
    

In [ ]:
# Final Summary
print('=' * 70)
print('SEC EDGAR FUNDAMENTALS SUMMARY')
print('=' * 70)

print('\n1. DATA SOURCE')
print('   All data fetched from SEC EDGAR API (data.sec.gov)')
print('   Uses XBRL company facts — the official SEC filing data')

print('\n2. COMPANIES ANALYZED')
for ticker, name in tickers.items():
    cik = get_cik(ticker) if False else 'cached'  # Don't re-fetch
    status = 'Data fetched' if ticker in company_data else 'Not available'
    print(f'   {ticker} — {name}: {status}')

print('\n3. METRICS EXTRACTED')
print('   - Annual Revenue (from 10-K filings)')
print('   - Annual Net Income')
print('   - Profit Margin (Net Income / Revenue)')
print('   - Earnings Per Share (EPS)')

print('\n4. IMPORTANT NOTES')
print('   - SEC EDGAR uses XBRL tags which vary across companies')
print('   - Some companies may use different tag names for the same metric')
print('   - The SEC API may rate-limit shared IPs (common in Colab)')
print('   - If data is missing, the company may use non-standard tags')

print('\n5. NO SYNTHETIC DATA')
print('   Every number comes directly from official SEC filings.')
print('   If the API fails, the notebook reports the error clearly.')

print('=' * 70)
print('All data is REAL from SEC EDGAR filings. No synthetic data used.')
print('=' * 70)